## Implement Lasso Regression using ISTA

In [1]:
import numpy as np

def soft_threshold(w: np.ndarray, threshold: float) -> np.ndarray:
    """Apply soft-thresholding operator element-wise.
    
    S(w, λ) = sign(w) * max(|w| - λ, 0)
    
    Args:
        w: Input array
        threshold: Threshold value λ
    
    Returns:
        Soft-thresholded array where:
        - Values with |w| > λ are shrunk toward zero by λ
        - Values with |w| ≤ λ become exactly zero
    """
    return np.sign(w) * np.maximum(np.abs(w) - threshold, 0)


def l1_regularization_gradient_descent(
    X: np.ndarray, 
    y: np.ndarray, 
    alpha: float = 0.1, 
    learning_rate: float = 0.01, 
    max_iter: int = 1000, 
    tol: float = 1e-4
) -> tuple:
    """
    Implement Lasso Regression using ISTA (Iterative Shrinkage-Thresholding Algorithm).
    
    ISTA alternates between:
    1. Gradient step on MSE loss: w_temp = w - lr * gradient_mse
    2. Proximal step (soft-thresholding): w_new = soft_threshold(w_temp, lr * alpha)
    
    Args:
        X: Feature matrix of shape (n_samples, n_features)
        y: Target vector of shape (n_samples,)
        alpha: L1 regularization strength
        learning_rate: Step size for gradient descent
        max_iter: Maximum iterations
        tol: Convergence tolerance on weight change
    
    Returns:
        tuple: (weights, bias)
    
    Note: The bias term is NOT regularized.
    """
    n_samples, n_features = X.shape
    weights = np.zeros((n_features, 1))
    bias = 0.0
    
    # Ensure y is column vector
    y = y.reshape(-1, 1)
    
    for _ in range(max_iter):
        # Forward pass
        y_pred = X @ weights + bias
        error = y_pred - y
        
        # Step 1: Gradient descent on MSE (smooth part)
        dw_mse = (1/n_samples) * (X.T @ error)
        db = (1/n_samples) * np.sum(error)
        
        w_temp = weights - learning_rate * dw_mse
        b_temp = bias - learning_rate * db
        
        # Step 2: Proximal step (soft-thresholding for L1)
        w_new = soft_threshold(w_temp, alpha * learning_rate)
        b_new = b_temp
        
        # Check convergence BEFORE updating
        weight_change = np.linalg.norm(w_new - weights, ord=2)
        
        # Update weights and bias
        weights = w_new
        bias = b_new
        
        # Break after update if converged
        if weight_change < tol:
            break
    
    return (weights.flatten(), bias)

In [2]:
X = np.array([[1, 0.01], [2, 0.02], [3, 0.03], [4, 0.04], [5, 0.05]])
y = np.array([2, 4, 6, 8, 10])

In [3]:
weights, bias = l1_regularization_gradient_descent(X, y, alpha=0.5, learning_rate=0.01, max_iter=1000)

In [4]:
weights, bias

(array([1.81057949, 0.        ]), np.float64(0.5258897372032391))